This is me trying to do endogenous regime change now. Hopefully it doesn't take that long? Idk sobbing this all feels exhausting wot am i doing

In [2]:
println("Adding Packages...\n")
using Pkg
Pkg.activate(".")   # so you're using the same packages as me
Pkg.instantiate(; verbose=true)

using JLD2
using Random
using Printf
using LinearAlgebra
using StatsBase: countmap
using Dates
using Revise

Adding Packages...



  Activating project at `f:\Dropbox\Education\OSU\Ongoing_Research\Populism\political-polarization\p`


Now importing the relevant code that I've written: putting this in a different block because that means I can reimport it as needed without much chaos. This takes advantage of the revise module, which checks whether the module has already been loaded and updates if so.

In [28]:
for file in ["src/ModelTypes.jl", "src/Compute.jl", "src/DistrTools.jl", "src/ModelFunctions.jl",
             "src/EGM.jl", "src/Solvers.jl", "src/SteadyState.jl", "src/Predict.jl"]
    modname = Symbol(splitext(basename(file))[1])       # taken from claude, not sure this symbol thing yet
    existed = isdefined(Main, modname)
    includet(file)
    println(existed ? "Updated $file" : "Loaded $file")
end

using .ModelTypes, .Compute, .EGM, .DistrTools, .Solvers, .SteadyState, .Predict, .ModelFunctions

Updated src/ModelTypes.jl
Updated src/Compute.jl
Updated src/DistrTools.jl
Updated src/ModelFunctions.jl
Updated src/EGM.jl
Updated src/Solvers.jl
Updated src/SteadyState.jl
Updated src/Predict.jl


Inititalizing the actual environment

In [5]:
# ─── Frequency-invariant parameters ───
const α::Float64 = 0.36
const σ::Float64 = 2
const ϕ::Float64 = 0
const μ_l::Float64 = 0
const μ_z::Float64 = 0

# grid sizes and parameters
const na::Int64 = 100; 
const nl::Int64 = 15;
const nz::Int64 = 2;
const nk::Int64 = 37;

const a_l::Float64 = 0;
const a_h::Float64 = 100;

# policy grids
const np::Int64 = 10; # number of policies
const pol_l::Float64 = 0;
const pol_h::Float64 = .18;

τ_grid = range(pol_l, pol_h, length = np);
η_grid = range(pol_l, pol_h, length = np);
policy_grid = [(η, τ) for η in η_grid, τ in τ_grid];
captax = repeat([0.0], outer = nl);

# ─── Frequency switch: read from environment, default to "annual" ───
const freq = get(ENV, "FREQ", "quarterly")
display("Frequency set to: $freq")

if freq == "quarterly"
    const β::Float64   = 0.99
    const δ::Float64   = 0.025
    const ρ_l::Float64 = 0.9878      # STY persistence quarterly
    const σ_l::Float64 = 0.087       # STY innovation std, quarterly
    const ρ_z::Float64 = 0.976
    const σ_z::Float64 = 0.007
    
			
elseif freq == "annual"
    const β::Float64   = 0.96
    const δ::Float64   = 0.06
    const ρ_l::Float64 = 0.952     # Storesletten-Telmer-Yaron
    const σ_l::Float64 = 0.17       # = sqrt(0.061), STY persistent innovation std σ_η
    const ρ_z::Float64 = 0.909      # Khan-Thomas 2013
    const σ_z::Float64 = 0.014
    
else
    error("FREQ must be \"annual\" or \"quarterly\", got \"$freq\"")
end

const freq_label = freq;   # use in output filename

"Frequency set to: quarterly"

Now building the grids

In [22]:
grid_range_l = 3.5     # wide, for near-unit-root labor
grid_range_z = 2.575   # leave z as is

π_l, lgrid = getTauchen(nl,  μ_l, σ_l, ρ_l, grid_range_l);
# π_z, zgrid= getTauchen(nz,  μ_z, σ_z, ρ_z, grid_range_z);

# stationary_l = stationary(π_l);
# const lagg::Float64 = dot(stationary_l, lgrid);

zgrid = [0.98, 1.02]          # bad, good
π_z = [0.80  0.20;                 # from bad: 20% chance exit to good
       0.03125  0.96875]          # from good: ~3% chance enter recession

nt = length(LinearIndices(π_z));       # number of states in joint markov chain

stationary_l = stationary(π_l);
const lagg::Float64 = dot(stationary_l, lgrid);
 
# more conservative estimates 
# const kH::Float64 = ((1.0/β - 1.0 + δ) / α)^(1.0/(α-1.0)) * (1.0 + maximum(η_grid)) * 1.5
# const kL::Float64 = max(1.0, ((1.0/β - 1.0 + δ) / α)^(1.0/(α-1.0)) * (1.0 + minimum(η_grid)) * 0.5)

const kL::Float64 = 1; const kH::Float64 = 50; 
Kgrid = collect(range(kL, kH, length = nk));

agrid = logspace(a_l, a_h, na);
amu = collect(range(a_l, a_h, length=na*10));

const NT = 5000; #three thousand periods for sampling
const rnseed = 1234567;

const zt = simz(NT, nz, rnseed, π_z);

const params = ModelParams(α, β, δ, σ, ϕ, agrid, 
		lgrid, zgrid, π_l, π_z, amu, Kgrid);

The next thing that needs to be done is like, figuring out what to do to make this endogenous. I'm doing the first part, which is just running it to make sure I can get the forecasts based on z-z transitions, ignoring regime change for now.

In [ ]:
Kfore_start = repeat([0.05 0.95], nt)

η = 0.0; τ = 0.0; captax = repeat([0.0], outer = nl); 

policies = ProposedPolicies(η, τ, captax);

r_vals = zeros(nk, nt); w_vals = zeros(nk, nt); λ_vals = zeros(nk, nt); 

for ik = 1:nk, it = 1:nt
	iz = CartesianIndices(π_z)[it][2] # getting today's z'
	r_vals[ik, it] = calcr(α, δ, Kgrid[ik], η, zgrid[iz])
	w_vals[ik, it] = calcw(α, Kgrid[ik], η, zgrid[iz])
	denom = dot((w_vals[ik, it] .* lgrid).^(1 - τ), stationary(π_l))
	tot_inc = w_vals[ik, it] * dot(lgrid, stationary(π_l))
	λ_vals[ik, it] = tot_inc / denom
end

prices = ImpliedRegimeParams_KS(λ_vals, r_vals, w_vals)

ImpliedRegimeParams_KS([1.0 1.0 0.9999999999999999 0.9999999999999999; 0.9999999999999998 0.9999999999999998 1.0 1.0; … ; 1.0000000000000002 1.0000000000000002 1.0 1.0; 0.9999999999999999 0.9999999999999999 0.9999999999999999 0.9999999999999999], [0.3278 0.3278 0.34219999999999995 0.34219999999999995; 0.17857950989528829 0.17857950989528829 0.18688887764611636 0.18688887764611636; … ; 0.004366963500782691 0.004366963500782691 0.005565615072243204 0.005565615072243204; 0.0038527872669087936 0.0038527872669087936 0.0050304520533132335 0.0050304520533132335], [0.6272 0.6272 0.6528 0.6528; 0.8545312761036792 0.8545312761036792 0.8894101036997478 0.8894101036997478; … ; 2.5393359550553325 2.5393359550553325 2.6429823205677954 2.6429823205677954; 2.5646922015030036 2.5646922015030036 2.6693735158500655 2.6693735158500655])

Now we start the actual solution part.

In [ ]:
# put this in again here since I'm testing and modifying regularly.

for file in ["src/ModelTypes.jl", "src/Compute.jl", "src/DistrTools.jl", "src/ModelFunctions.jl",
             "src/EGM.jl", "src/Solvers.jl", "src/SteadyState.jl", "src/Predict.jl"]
    modname = Symbol(splitext(basename(file))[1])       # taken from claude, not sure this symbol thing yet
    existed = isdefined(Main, modname)
    includet(file)
    println(existed ? "Updated $file" : "Loaded $file")
end

# init V and friends:
V0 = zeros(nk,nt,nl,na); V  = zeros(nk,nt,nl,na);
G0 = zeros(nk,nt,nl,na); G  = zeros(nk,nt,nl,na);  
C  = zeros(nk,nt,nl,na);

# intializing a starting guess
for ik = 1:nk, it = 1:nt, il = 1:nl, ia = 1:na
	kval = agrid[ia];
	yval = (1 + r_vals[ik, it]*(1-captax[il]))*kval + w_vals[ik, it]*lgrid[il] - r_vals[ik, it]*ϕ;
	ymin = max(1e-10, yval);
	V0[ik, it, il, ia] = log(ymin);
	G0[ik, it, il, ia] = agrid[ia];
end

const dTol = 1e-4;
const vTol = 1e-4;

Kfore_out, Kt, it_t = run_KS(V, V0, G, G0, C, params, policies, prices,
			zt, Kfore_start, vTol, dTol, λ_damp=0.5, verbose = true)

Updated src/ModelTypes.jl
Updated src/Compute.jl
Updated src/DistrTools.jl
Updated src/ModelFunctions.jl
Updated src/EGM.jl
Updated src/Solvers.jl
Updated src/SteadyState.jl
Updated src/Predict.jl
Solving Household Problem...
	Iteration 100: ||V - V0|| = 0.040031, ||G - G0|| = 0.000001, dist = 0.040031
	Iteration 200: ||V - V0|| = 0.014317, ||G - G0|| = 0.000000, dist = 0.014317
	Iteration 300: ||V - V0|| = 0.005214, ||G - G0|| = 0.000000, dist = 0.005214
	Iteration 400: ||V - V0|| = 0.001904, ||G - G0|| = 0.000000, dist = 0.001904
	Iteration 500: ||V - V0|| = 0.000696, ||G - G0|| = 0.000000, dist = 0.000696
	Iteration 600: ||V - V0|| = 0.000255, ||G - G0|| = 0.000000, dist = 0.000255
	Iteration 700: ||V - V0|| = 0.000093, ||G - G0|| = 0.000000, dist = 0.000093
	Iteration 800: ||V - V0|| = 0.000034, ||G - G0|| = 0.000000, dist = 0.000034
	Iteration 900: ||V - V0|| = 0.000012, ||G - G0|| = 0.000000, dist = 0.000012
	Iteration 1000: ||V - V0|| = 0.000005, ||G - G0|| = 0.000000, dist = 0.